# Setup

### Libraries

In [ ]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt

# Settings
import warnings
warnings.filterwarnings("ignore")

# Define settings for graphics
plt.style.use('dark_background')

# Current working directory
print(os.getcwd())

### Read Data to Dataframe
#### Nicht ausführbar, da die Dateien fehlen

In [ ]:
#Einzeln:
# Read apartment data
df = pd.read_csv('Data/apartments_data_prepared.csv', sep=',')
df.head(5)


#Ausführlicher:

# Define path to data
path = 'Data/'

# Read data from .csv files
df_customers = pd.read_csv(f'{path}Customers.csv', sep=',')
df_categories = pd.read_csv(f'{path}Categories.csv', sep=',')
df_employees = pd.read_csv(f'{path}Employees.csv', sep=',')
df_order_details = pd.read_csv(f'{path}OrderDetails.csv', sep=',')
df_orders = pd.read_csv(f'{path}Orders.csv', sep=',')
df_products = pd.read_csv(f'{path}Products.csv', sep=',')
df_shippers = pd.read_csv(f'{path}Shippers.csv', sep=',')
df_suppliers = pd.read_csv(f'{path}Suppliers.csv', sep=',')

# Change column names to lowercase
df_customers.columns = df_customers.columns.str.lower()
df_categories.columns = df_categories.columns.str.lower()
df_employees.columns = df_employees.columns.str.lower()
df_order_details.columns = df_order_details.columns.str.lower()
df_orders.columns = df_orders.columns.str.lower()
df_products.columns = df_products.columns.str.lower()
df_shippers.columns = df_shippers.columns.str.lower()
df_suppliers.columns = df_suppliers.columns.str.lower()

# Check dimensions of DataFrames
print("Dimensions of DataFrames:")
print(f"Customers: {df_customers.shape}")
print(f"Categories: {df_categories.shape}")
print(f"Employees: {df_employees.shape}")
print(f"OrderDetails: {df_order_details.shape}")
print(f"Orders: {df_orders.shape}")
print(f"Products: {df_products.shape}")
print(f"Shippers: {df_shippers.shape}")
print(f"Suppliers: {df_suppliers.shape}")

### Create Database Connection

In [ ]:
# Set up database connection
user = "pgadmin"
password = "geheim"
host = "localhost"
port = "5432"
database = "northwind"

# Create Connection URL
db_connection_url = "postgresql://" + user + ":" + password +\
                    "@" + host + ":" + port + "/" + database

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Test database connection
with engine.connect() as connection:
    result = connection.execute(text('SELECT current_database()'))
    print(result.fetchone())

# Dispose the engine
engine.dispose()

### Write data frames to database tables

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Drop tables if it exist
with engine.connect() as connection:
    connection.execute(text("DROP TABLE IF EXISTS public.categories CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.customers CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.employees CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.orderdetails CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.orders CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.products CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.shippers CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.suppliers CASCADE;"))
    connection.commit()

# Write DataFrames to database
df_categories.to_sql('categories', engine)
df_customers.to_sql('customers', engine)
df_employees.to_sql('employees', engine)
df_order_details.to_sql('orderdetails', engine)
df_orders.to_sql('orders', engine)
df_products.to_sql('products', engine)
df_shippers.to_sql('shippers', engine)
df_suppliers.to_sql('suppliers', engine)

# Dispose the engine
engine.dispose()

### List tables in the database

In [ ]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Open a connection
with engine.connect() as connection:

    # Execute the query
    result = connection.execute(text("""SELECT table_name
                                        FROM information_schema.tables
                                        WHERE table_schema = 'public';"""))
    
    # Fetch and print the results
    for row in result:
        print(row[0])

# Dispose the engine
engine.dispose()

### Create a SQL query to retrieve product information from a database.

In [ ]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Write data to table
df_sub = pd.read_sql_query('''SELECT 
                                Products.ProductID, 
                                Products.ProductName, 
                                Categories.CategoryName,
                                Products.Price
                             FROM Products
                             INNER JOIN Categories ON Products.CategoryID = Categories.CategoryID
                             ORDER BY Products.ProductID;''', 
                          con=engine)

# Dispose the engine
engine.dispose()

# Show the data
df_sub

### Plot product prices

In [ ]:
# Plot Histogram
fig = plt.figure( figsize=(7,4))
plt.xticks(fontsize=14, rotation=0)
plt.yticks(fontsize=14, rotation=0)
n, bins, patches = plt.hist(x=df_sub['price'], 
                            bins=20, 
                            color='#C53115',
                            alpha=1.00, 
                            rwidth=0.95
                   )
plt.grid(alpha=0.25)
plt.ticklabel_format(style='plain')

# Set labels
plt.xlabel('Price (USD)', fontsize=10, labelpad=10)
plt.ylabel('Frequency', fontsize=10, labelpad=10)
plt.title('Histogram of product prices', fontsize=12, pad=10)

# Set fontsize of tick labels
plt.xticks(fontsize = 10)
plt.yticks(fontsize = 10)

plt.show()